# v8 代替戦略: 高品質1000件データセット（Person Wスタイル）

## 戦略概要

**段階的SFTの代替戦略**として、Person Wの知見に基づく「少量高品質データ」アプローチ。

### Person Wの知見
```
ポイント:
- 最終的に1000件以下のデータで学習
- ハイパラの大きな変更より微調整
- epoch2、T4で40分程度
- データの質（今回のコンペに適切か）が重要
```

### v8_curated_1kデータセットの特徴
- **サンプル数**: 1000件以下の厳選データ
- **品質基準**:
  - パース成功率100%
  - コードフェンス混入なし
  - 説明文混入なし
  - 適切な複雑さ
- **フォーマット**: 全5種類（JSON/CSV/YAML/XML/TOML）バランス配分

### 段階的SFTとの比較

| 項目 | 段階的SFT (Stage 1-4) | Person Wスタイル (本ノートブック) |
|------|----------------------|--------------------------------|
| データ量 | 合計2800件 | 1000件以下 |
| 学習回数 | 4段階 | 1回 |
| 所要時間 | 長い（4回分） | 短い（1回分） |
| 複雑さ | 高い | シンプル |
| リスク | Stage間の干渉 | オーバーフィット |

### 使用タイミング
- 段階的SFT（Stage 1-4）の結果が期待通りでない場合
- シンプルなアプローチで再実験したい場合
- 時間が限られている場合

## Step 1: 依存関係インストール

In [ ]:
!pip -q uninstall -y numpy pandas datasets trl transformers accelerate peft unsloth unsloth-zoo bitsandbytes xformers
!pip -q install "numpy==2.0.2" "pandas==2.2.2"
!pip -q install \
  "datasets==4.3.0" \
  "trl==0.24.0" \
  "transformers==4.56.2" \
  "accelerate==1.1.0" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.0"
!pip -q install "unsloth-zoo==2025.12.7" "unsloth==2025.12.7"

In [ ]:
import numpy as np, pandas as pd
import datasets, trl, transformers, torch
from unsloth import FastLanguageModel

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("unsloth import OK")

## Step 2: 環境変数設定（v8_curated_1k用）

**Person Wスタイルの設定:**
- ベースモデル: Qwen3-4B-Instruct-2507（オリジナルから開始）
- epoch: 2
- 学習率: 控えめ（5e-5）

In [ ]:
import os

VERSION = "v8_curated_1k"
TITLE_LINE = f"qwen3-4b-structured-output-lora-{VERSION}"

def _getenv(name: str, default: str):
    return os.environ.get(name, default)

def _getenv_int(name: str, default: int) -> int:
    try:
        return int(os.environ.get(name, str(default)))
    except Exception:
        return default

def _getenv_float(name: str, default: float) -> float:
    try:
        return float(os.environ.get(name, str(default)))
    except Exception:
        return default

# ============================================================
# v8_curated_1k 設定（Person Wスタイル）
# ============================================================

# ベースモデル: オリジナルのQwen3-4Bから開始（段階的SFTではない）
os.environ["SFT_BASE_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["SFT_DATASET_ID"] = "LOCAL:/content/train.json"  # v8_curated_1k
os.environ["SFT_OUT_LORA_DIR"] = f"/content/lora_structeval_{VERSION}"

os.environ["SFT_SEED"] = "3407"
os.environ["SFT_VAL_RATIO"] = "0.05"
os.environ["SFT_MAX_SEQ_LEN"] = "1024"

os.environ["SFT_LORA_R"] = "64"
os.environ["SFT_LORA_ALPHA"] = "128"
os.environ["SFT_LORA_DROPOUT"] = "0"
os.environ["SFT_LORA_TARGET_MODULES"] = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"

# Person Wスタイル: epoch2、標準的な学習率
os.environ["SFT_EPOCHS"] = "2"
os.environ["SFT_PER_DEVICE_TRAIN_BS"] = "2"
os.environ["SFT_PER_DEVICE_EVAL_BS"] = "2"
os.environ["SFT_GRAD_ACCUM"] = "8"
os.environ["SFT_LR"] = "5e-5"
os.environ["SFT_WARMUP_RATIO"] = "0.1"
os.environ["SFT_WEIGHT_DECAY"] = "0.05"

os.environ["SFT_MAX_STEPS"] = "-1"
os.environ["SFT_LOGGING_STEPS"] = "10"
os.environ["SFT_EVAL_STEPS"] = "50"
os.environ["SFT_SAVE_STEPS"] = "100"
os.environ["SFT_SAVE_TOTAL_LIMIT"] = "2"

os.environ["SFT_MASK_COT"] = "1"
os.environ["SFT_OUTPUT_MARKERS"] = "Output:,OUTPUT:,Final:,Answer:,Result:,Response:"
os.environ["SFT_OUTPUT_LEARN_MODE"] = "after_marker"
os.environ["SFT_USE_UPSAMPLING"] = "0"
os.environ["SFT_UPSAMPLE_RULES"] = '{}'

print(f'=== v8 代替戦略: Person Wスタイル ===')
print(f'  データセット: {VERSION} (1000件以下の高品質データ)')
print(f'  ベースモデル: Qwen/Qwen3-4B-Instruct-2507')
print(f'  MAX_SEQ_LEN={os.environ["SFT_MAX_SEQ_LEN"]}, EPOCHS={os.environ["SFT_EPOCHS"]}, LR={os.environ["SFT_LR"]}')
print(f'  目標: 全フォーマット高精度 → LB 0.8+')

## Step 3: HuggingFace ログイン

In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi

print("Attempting login...")
MY_TOKEN = userdata.get("HF_TOKEN")
try:
    if MY_TOKEN:
        login(token=MY_TOKEN.strip())
        api = HfApi()
        print("✅ Login successful")
    else:
        raise ValueError("HF_TOKEN is empty")
except Exception as e:
    print(f"⚠️ Auto-login failed: {e}")
    login()
    api = HfApi()

## Step 4: WandB（Weights & Biases）ログイン

In [ ]:
import wandb

WANDB_KEY = userdata.get("WANDB_API_KEY")

try:
    if WANDB_KEY:
        wandb.login(key=WANDB_KEY.strip())
        print("✅ WandB login successful")
    else:
        raise ValueError("WANDB_API_KEY is empty")
except Exception as e:
    print(f"⚠️ Auto-login failed: {e}")
    wandb.login()

wandb.init(
    project="structeval-sft",
    name=VERSION,
    config={
        "strategy": "person_w_style",
        "base_model": os.environ.get("SFT_BASE_MODEL"),
        "epochs": os.environ.get("SFT_EPOCHS"),
        "learning_rate": os.environ.get("SFT_LR"),
        "lora_r": os.environ.get("SFT_LORA_R"),
        "max_seq_len": os.environ.get("SFT_MAX_SEQ_LEN"),
        "data_size": "<= 1000",
    }
)
print(f"✅ WandB initialized: {wandb.run.name}")

## Step 5: データセットのアップロード

**アップロードするファイル:**
`inputs/sft_processed/v8_curated_1k/train.json`

In [ ]:
import json
import shutil
from google.colab import files
from collections import Counter

print("train.json (v8_curated_1k) をアップロードしてください...")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.json'):
        target_path = '/content/train.json'
        if filename != 'train.json':
            shutil.move(filename, target_path)
        print(f"✅ アップロード完了: {target_path}")
        with open(target_path, 'r') as f:
            data = json.load(f)
        print(f"   データ件数: {len(data)} 件")
        
        # フォーマット分布確認
        formats = []
        for sample in data:
            if 'metadata' in sample and isinstance(sample['metadata'], dict):
                fmt = sample['metadata'].get('format', 'unknown')
                formats.append(fmt)
        print(f"   フォーマット分布: {Counter(formats)}")
        break

## Step 6: 学習の実行

In [ ]:
import random
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, Trainer

BASE_MODEL_ID = _getenv("SFT_BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
DATASET_ID    = _getenv("SFT_DATASET_ID", "LOCAL:/content/train.json")
OUT_LORA_DIR  = _getenv("SFT_OUT_LORA_DIR", "/content/lora")
SEED        = _getenv_int("SFT_SEED", 3407)
VAL_RATIO   = _getenv_float("SFT_VAL_RATIO", 0.05)
MAX_SEQ_LEN = _getenv_int("SFT_MAX_SEQ_LEN", 1024)
LORA_R       = _getenv_int("SFT_LORA_R", 64)
LORA_ALPHA   = _getenv_int("SFT_LORA_ALPHA", 128)
LORA_DROPOUT = _getenv_float("SFT_LORA_DROPOUT", 0)
LORA_TARGET_MODULES = _getenv("SFT_LORA_TARGET_MODULES", "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj").split(",")
NUM_TRAIN_EPOCHS = _getenv_int("SFT_EPOCHS", 2)
PER_DEVICE_TRAIN_BATCH_SIZE = _getenv_int("SFT_PER_DEVICE_TRAIN_BS", 2)
PER_DEVICE_EVAL_BATCH_SIZE  = _getenv_int("SFT_PER_DEVICE_EVAL_BS", 2)
GRAD_ACCUM   = _getenv_int("SFT_GRAD_ACCUM", 8)
LR           = _getenv_float("SFT_LR", 5e-5)
WARMUP_RATIO = _getenv_float("SFT_WARMUP_RATIO", 0.1)
WEIGHT_DECAY = _getenv_float("SFT_WEIGHT_DECAY", 0.05)
MAX_STEPS    = _getenv_int("SFT_MAX_STEPS", -1)
LOGGING_STEPS = _getenv_int("SFT_LOGGING_STEPS", 10)
EVAL_STEPS   = _getenv_int("SFT_EVAL_STEPS", 50)
SAVE_STEPS   = _getenv_int("SFT_SAVE_STEPS", 100)
SAVE_TOTAL_LIMIT = _getenv_int("SFT_SAVE_TOTAL_LIMIT", 2)

print(f"BASE_MODEL_ID: {BASE_MODEL_ID}")
print(f"Strategy: Person W Style (High-quality <= 1000 samples)")

In [ ]:
# データセット読み込み
def load_sft_dataset(dataset_id: str):
    if dataset_id.startswith("LOCAL:"):
        local_path = dataset_id.replace("LOCAL:", "")
        with open(local_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return Dataset.from_list(data)
    else:
        return load_dataset(dataset_id, split="train")

raw_ds = load_sft_dataset(DATASET_ID)
print(f"Loaded {len(raw_ds)} samples")

random.seed(SEED)
indices = list(range(len(raw_ds)))
random.shuffle(indices)
val_size = int(len(raw_ds) * VAL_RATIO)
train_ds = raw_ds.select(indices[val_size:])
val_ds = raw_ds.select(indices[:val_size])

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# モデルロード
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded with LoRA")

In [ ]:
# トークン化
def preprocess_function(examples):
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in examples["messages"]]
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_SEQ_LEN, padding=False, return_tensors=None)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
print(f"Tokenized: train={len(tokenized_train)}, val={len(tokenized_val)}")

In [ ]:
# 学習
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)

training_args = TrainingArguments(
    output_dir=OUT_LORA_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    max_steps=MAX_STEPS,
    seed=SEED,
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    report_to="wandb",
    run_name=VERSION,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(OUT_LORA_DIR)
tokenizer.save_pretrained(OUT_LORA_DIR)
print(f"✅ Saved to {OUT_LORA_DIR}")

# WandBセッション終了
if wandb.run is not None:
    wandb.finish()
    print("✅ WandBセッションを終了しました")

## Step 7: モデルをHugging Faceにアップロード

In [ ]:
from pathlib import Path
from peft import PeftModel

LORA_SAVE_DIR = Path(OUT_LORA_DIR)
HF_REPO_ID = _getenv("HF_REPO_ID", f"kmd2525/{VERSION}-merged")
PRIVATE = True

del model, trainer
torch.cuda.empty_cache()

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

model = PeftModel.from_pretrained(base_model, str(LORA_SAVE_DIR))
model = model.merge_and_unload()

STAGE_DIR = Path("/content/hf_upload_curated")
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(STAGE_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(STAGE_DIR))
print(f"📦 Model saved to {STAGE_DIR}")

In [ ]:
# Model Card (README.md) を作成
model_card_content = f'''---
license: apache-2.0
base_model: {BASE_MODEL_ID}
tags:
  - structured-output
  - json
  - csv
  - yaml
  - xml
  - toml
  - sft
  - high-quality-data
language:
  - en
  - ja
---

# {VERSION}-merged

## Model Description

This model is trained using the **Person W Style** approach - high-quality data under 1000 samples.

### Training Strategy (Person W Style)

Based on Person W\'s insight that achieved 0.8+ with:
- High-quality data under 1000 samples
- epoch=2
- Fine-tuning hyperparameters rather than large changes

### Dataset: v8_curated_1k

**Quality Criteria:**
- 100% parse success rate
- No code fence contamination
- No natural language prefix/suffix
- Appropriate complexity

**Format Distribution:**
- Balanced across all 5 formats (JSON/CSV/YAML/XML/TOML)

### Training Parameters

- MAX_SEQ_LEN: {MAX_SEQ_LEN}
- EPOCHS: {NUM_TRAIN_EPOCHS}
- Learning Rate: {LR}
- LoRA R: {LORA_R}, Alpha: {LORA_ALPHA}

### Comparison with Sequential Format Learning

| Approach | Data | Training | Complexity |
|----------|------|----------|------------|
| Sequential SFT (Stage 1-4) | 2800 samples | 4 stages | High |
| Person W Style (This model) | < 1000 samples | 1 stage | Simple |

### Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}")
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")
```
'''

readme_path = STAGE_DIR / "README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(model_card_content)
print(f"✅ Model Card saved: {readme_path}")

api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=PRIVATE)
api.upload_folder(folder_path=str(STAGE_DIR), repo_id=HF_REPO_ID, repo_type="model", commit_message=f"Upload {VERSION}")

print(f"✅ Uploaded: https://huggingface.co/{HF_REPO_ID}")

## 🎉 v8_curated_1k 完了!

### 段階的SFTとの比較

| 実験 | データ量 | アプローチ | LBスコア |
|------|---------|----------|----------|
| v8_stage4 (Sequential SFT) | 2800件 | 4段階学習 | ??? |
| **v8_curated_1k (Person W)** | **<1000件** | **1回学習** | **???** |

### 推論とLB提出

推論ノートブックでこのモデルを使用:
```python
MODEL_ID = "kmd2525/v8_curated_1k-merged"
```

## Step 8: Colabインスタンスの削除（課金停止）

In [ ]:
# ============================================================
# Colabインスタンスの削除（課金停止）
# ============================================================
# 以下を実行するとランタイムが終了し、インスタンスが削除されます
# 必ずHFへのアップロードが完了してから実行してください！

from google.colab import runtime
runtime.unassign()